## Dans ce notebook, nous verrons :
1. L'influence de l'apprentissage sur plusieurs capteurs
2. Comment entrainer notre modèle à débruiter un signal : on passera d'un Auto-Encodeur (AE) à un Denoising Auto-Encodeur (DAE)

In [1]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features, models


In [2]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix
from keras.callbacks import EarlyStopping
from Sources import data, config, features, models

DUREE_FENETRE = config.DUREE_FENETRE

def construire_fenetres_brutes(selection, canal="DE", duree=DUREE_FENETRE/2, recouvrement=0.5):
    SB, lignes = [], []
    for nom, info in selection.iterrows():
        s = data.charger(nom, canaux=(canal,))
        for i, val in enumerate(features.fenetrer(s[canal], s["fs"], duree=duree, recouvrement=recouvrement)):
            SB.append(val)
            lignes.append({"fichier": nom, "fenetre": i, "canal": canal,
                           "defaut": info["defaut"], "charge_hp": info["charge_hp"],
                           "rpm": s["rpm"], "fs": s["fs"],
                           "defaut_sain": "Sain" if info["defaut"] == "Sain" else "Défaut",
                           "diametre_mils": info["diametre_mils"],
                           "classe": "Sain" if info["defaut"] == "Sain" else info["defaut"] + str(info["diametre_mils"]).zfill(3)})
    return np.array(SB, dtype=np.float32), pd.DataFrame(lignes)

def score_mse(modele, X):
    Xr = modele.predict(X[..., None], verbose=0)[..., 0]
    return ((X - Xr) ** 2).mean(axis=1)

def ajouter_bruit(x, snr_db, seed=42):
    rng = np.random.default_rng(seed)
    p = np.mean(x ** 2)
    return x + np.sqrt(p / 10 ** (snr_db / 10)) * rng.standard_normal(x.shape)

# --- DE + FE ---
selection_DE = data.selectionner(sources=("12kDE", "Normal"), defauts=("IR","B","OR"),
                                 diametres=(7,14,21), positions_or=("6",), charges=(0,1,2,3),
                                 canaux={"12kDE": ("DE",), "Normal": ("DE",)})
selection_FE = data.selectionner(sources=("12kFE", "Normal"), defauts=("IR","B","OR"),
                                 diametres=(7,14,21), positions_or=("3","6","12"), charges=(0,1,2,3),
                                 canaux={"12kFE": ("FE",), "Normal": ("FE",)})
SB_DE, infos_DE = construire_fenetres_brutes(selection_DE, canal="DE")
SB_FE, infos_FE = construire_fenetres_brutes(selection_FE, canal="FE")

SB = np.concatenate([SB_DE, SB_FE], axis=0)
infos = pd.concat([infos_DE, infos_FE], ignore_index=True)
SBNorm = StandardScaler().fit_transform(SB.T).T
print(SB.shape); print(infos.groupby(["canal", "defaut_sain"]).size())

(21944, 1200)
canal  defaut_sain
DE     Défaut         7269
       Sain           2825
FE     Défaut         9025
       Sain           2825
dtype: int64


In [ ]:
groups = infos["fichier"] + "_" + infos["canal"]          # "Normal_L0_DE" ≠ "Normal_L0_FE"
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=40)
i_train, i_test = next(gss.split(infos, groups=groups))

sain = (infos["defaut_sain"] == "Sain").to_numpy()

i_sain_FE = np.where((infos["canal"] == "FE") & sain)[0]
i_test = np.union1d(i_test, i_sain_FE[: len(i_sain_FE) // 4])   # 25 % des sains FE au test
i_train = np.setdiff1d(np.arange(len(infos)), i_test)
X_train_sain = SBNorm[i_train][sain[i_train]]        # sains DE + FE du train

#Ce que fait la cellule
#groups associe chaque fenêtre à un groupe « fichier + capteur ». Normal_L0 vu par DE et Normal_L0 vu par FE forment ainsi deux groupes distincts.
#GroupShuffleSplit tire au hasard 25 % des groupes pour le test. Toutes les fenêtres d'un même fichier vont donc du même côté : la séparation est propre.
#Ensuite, 25 % des fenêtres saines FE sont ajoutées au test : i_sain_FE[: len(i_sain_FE) // 4].
#X_train_sain regroupe toutes les fenêtres saines restantes, DE et FE.
#Le problème de l'étape 3

#L'étape 3 prend les 706 premières fenêtres saines FE dans l'ordre, sans tenir compte des fichiers. Or :

#Normal_L0 donne environ 405 fenêtres ;
#Normal_L1, Normal_L2 et Normal_L3 en donnent environ 806 chacun.

#Une coupure à 706 tombe donc au milieu d'un fichier. Une partie de ses fenêtres va en test, le reste en apprentissage. Avec 50 % de recouvrement, deux fenêtres voisines partagent même la moitié de leurs points. C'est la fuite de données

In [3]:
canal = infos["canal"].to_numpy()
charge = infos["charge_hp"].to_numpy()
sain = (infos["defaut_sain"] == "Sain").to_numpy()

# Apprentissage : uniquement les sains DE à 0, 1 et 2 HP (comme pour le RMS, notebook 08)
apprentissage = (canal == "DE") & sain & (charge <= 2)
i_train = np.where(apprentissage)[0]

# Test DE : le sain DE à 3 HP et tous les défauts DE
i_test_DE = np.where((canal == "DE") & ~apprentissage)[0]

# Test FE : tout le capteur FE (sains et défauts), jamais vu à l'apprentissage
i_test_FE = np.where(canal == "FE")[0]

X_train_sain = SBNorm[i_train]
print(len(i_train), len(i_test_DE), len(i_test_FE))

2017 8077 11850


In [4]:
i_test = np.concatenate([i_test_DE, i_test_FE])      # tout le test, pour le contrôle

pd.DataFrame({
    "train": infos.iloc[i_train][["defaut_sain", "canal"]].value_counts(),
    "test":  infos.iloc[i_test][["defaut_sain", "canal"]].value_counts(),
}).fillna(0).astype(int)

train  test
defaut_sain canal             
Défaut      DE         0  7269
            FE         0  9025
Sain        DE      2017   808
            FE         0  2825

In [7]:
def evaluer_par_capteur(modele, tag):
    score = score_mse(modele, SBNorm)                          # score d'anomalie de chaque fenêtre
    seuil = np.percentile(score[i_train], 99)                  # seuil : p99 des sains DE d'apprentissage
    y = (infos["defaut_sain"].to_numpy() == "Défaut").astype(int)
    groupes = {"DE : sain 3 HP + défauts": i_test_DE,
               "FE : toutes charges": i_test_FE,
               "FE : 3 HP": i_test_FE[charge[i_test_FE] == 3]}
    lignes = []
    for nom, idx in groupes.items():
        TN, FP, FN, TP = confusion_matrix(y[idx], score[idx] > seuil, labels=[0, 1]).ravel()
        lignes.append({"config": tag, "test": nom, "seuil_p99": round(float(seuil), 4),
                       "AUC": round(roc_auc_score(y[idx], score[idx]), 3),
                       "fausses alertes (%)": round(100 * FP / (FP + TN), 2),
                       "défauts manqués (%)": round(100 * FN / (FN + TP), 2)})
    return pd.DataFrame(lignes)

modele_AE = models.conv1d_ae(SBNorm.shape[1])
modele_AE.fit(X_train_sain[..., None], X_train_sain[..., None], epochs=50, batch_size=64,
              validation_split=0.1, verbose=1,
              callbacks=[EarlyStopping(monitor="val_loss", patience=5, min_delta=0.02,
                                       restore_best_weights=True)])
df_AE = evaluer_par_capteur(modele_AE, "AE (sains DE seuls)")
display(df_AE)

Epoch 1/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 5s 121ms/step - loss: 0.6173 - val_loss: 0.3029
Epoch 2/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - loss: 0.1734 - val_loss: 0.1925
Epoch 3/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - loss: 0.1130 - val_loss: 0.0948
Epoch 4/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 0.0455 - val_loss: 0.0263
Epoch 5/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - loss: 0.0213 - val_loss: 0.0190
Epoch 6/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - loss: 0.0167 - val_loss: 0.0162
Epoch 7/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - loss: 0.0147 - val_loss: 0.0145
Epoch 8/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - loss: 0.0135 - val_loss: 0.0135
Epoch 9/50
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.0125 - val_loss: 0.0125


,config,test,seuil_p99,AUC,fausses alertes (%),défauts manqués (%)
0,AE (sains DE seuls),DE : sain 3 HP + défauts,0.0405,1.0,1.24,0.0
1,AE (sains DE seuls),FE : toutes charges,0.0405,1.0,3.93,0.0
2,AE (sains DE seuls),FE : 3 HP,0.0405,1.0,5.69,0.0
